# Biler (Maskinelæring & Regression-, klassifikation- & Clustering modeller)
Opgaven findes under "assignment" folderen

 ### Data loading

In [2]:
import urllib.request
import pandas as pd

# Download fra url
urllib.request.urlretrieve("https://raw.githubusercontent.com/foxdocs/dat-bi/refs/heads/main/Data/cars.csv", "data/cars.csv")

# Læs til dataframe
rawdataDF = pd.read_csv("data/cars.csv")
rawdataDF.head()

,Unnamed: 0,mpg,milage,maketype,car_id,region,price,make,type,scrapedate,alder,ccm2,doors,aldercat,navn
0,0,21.6,18000.0,"Toyota Yaris 1,5 VVT-iE T3 Smart 5d",5559177,København,169900.0,Toyota,Yaris,2022-11-08 22:29:38.772658,3,1500.0,5d,veteran,Region Hovedstaden
1,1,20.4,254000.0,"Ford Ka 1,2 Trend 3d",4367512,Nordjylland,34900.0,Ford,Ka,2022-11-08 22:29:38.772658,11,1200.0,3d,veteran,Region Nordjylland
2,2,NaN,51000.0,"Ford Mustang 4,7 V8 289cui. 2d",5506605,København,245000.0,Ford,Mustang,2022-11-08 22:29:38.772658,57,4700.0,2d,veteran,Region Hovedstaden
3,3,13.9,246000.0,"Toyota Avensis 1,8 VVT-i Sol stc. 5d",5518799,København,29999.0,Toyota,Avensis,2022-11-08 22:29:38.772658,18,1800.0,5d,veteran,Region Hovedstaden
4,4,17.9,97000.0,"Volvo V40 2,0 T2 122 Momentum Eco 5d",5604536,Nordjylland,199900.0,Volvo,V40,2022-11-08 22:29:38.772658,5,2000.0,5d,veteran,Region Nordjylland


### Data wrangling and exploration

In [8]:
# 1
## Ny dataframe
cleanDF = rawdataDF.copy()

# Dropper irrelevante kollonner
cleanDF = cleanDF.drop(columns=['Unnamed: 0', 'car_id', 'scrapedate', 'navn'])

# Retter taste fejl "ccm2"
cleanDF = cleanDF.rename(columns={'ccm2': 'ccm'})

# Fjerner "d" fra eksempelvis "5d" - gør beregning nemmere
cleanDF['doors'] = cleanDF['doors'].str.replace('d', '').astype(int)

# Fjerner missing values
cleanDF['mpg'] = cleanDF['mpg'].fillna(cleanDF['mpg'].median())
cleanDF['ccm'] = cleanDF['ccm'].fillna(cleanDF['ccm'].median())
cleanDF = cleanDF.dropna(subset=['region'])

# 2
# Finder hvilke features på en bil som har mest indflydelse på pris
numericDF = cleanDF.select_dtypes(include=['number'])
correlations = numericDF.corr()['price'].sort_values(ascending=False)
print(correlations)

cleanDF.head()

price     1.000000
ccm       0.257201
doors    -0.001846
mpg      -0.075939
alder    -0.343637
milage   -0.533661
Name: price, dtype: float64


,mpg,milage,maketype,region,price,make,type,alder,ccm,doors,aldercat
0,21.6,18000.0,"Toyota Yaris 1,5 VVT-iE T3 Smart 5d",København,169900.0,Toyota,Yaris,3,1500.0,5,veteran
1,20.4,254000.0,"Ford Ka 1,2 Trend 3d",Nordjylland,34900.0,Ford,Ka,11,1200.0,3,veteran
2,20.8,51000.0,"Ford Mustang 4,7 V8 289cui. 2d",København,245000.0,Ford,Mustang,57,4700.0,2,veteran
3,13.9,246000.0,"Toyota Avensis 1,8 VVT-i Sol stc. 5d",København,29999.0,Toyota,Avensis,18,1800.0,5,veteran
4,17.9,97000.0,"Volvo V40 2,0 T2 122 Momentum Eco 5d",Nordjylland,199900.0,Volvo,V40,5,2000.0,5,veteran


### Supervised machine learning by regression

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Definer independent (X) og dependent (y) variabler
X = cleanDF[['milage', 'alder', 'ccm', 'mpg']]
y = cleanDF['price']

# Vi splitter data 80% til træning og 20% til test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Træner regressionsmodel
model = LinearRegression()
model.fit(X_train, y_train)

# Forudsigelser
y_pred = model.predict(X_test)

# Matematiske formler til give maskinlæringen en score (Dette er med AI hjælp)
# Koden bruges til at se, om modellen rammer plet eller er helt væk:
# - MAE: Hvor mange kroner den i gennemsnit rammer ved siden af.
# - RMSE: Afslører hvis modellen laver nogle kæmpe, ekstreme fejl på enkelte biler.
# - R2: Karakter fra 0 til 1, der viser hvor godt alder, km og motor egentlig forklarer prisen.
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Printer resultaterne
print("--- Evaluation ----")
print("Gennemsnitlig fejl (MAE):", round(mae), "kr.")
print("Største fejl-straf (RMSE):", round(rmse), "kr.")
print("Samlet score (R2):", round(r2, 2))

--- Evaluation ----
Gennemsnitlig fejl (MAE): 48883 kr.
Største fejl-straf (RMSE): 66596 kr.
Samlet score (R2): 0.61


### Unsupervised machine learning for clustering


In [15]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Hvad vi går ud fra til vores clustering
features = ['price', 'milage', 'alder', 'ccm', 'mpg']
X_clustering = cleanDF[features]

# Skalering på data (Så alle mine features er lige)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clustering)

# Finder den bedste score
best_score = -1
best_k = 0

print("---Clustering Evaluation---")

for k in range(2, 7):
    # Træning af model
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, cluster_labels)
    print(f"Clusters: {k} | Score: {score:.3f}")
    
    # Gemmer den bedste konfiguration
    if score > best_score:
        best_score = score
        best_k = k

# Evaluation
print("\n---Anbefaling---")
print(f"The best configuration is {best_k} clusters with a score of {best_score:.3f}")

---Clustering Evaluation---
Clusters: 2 | Score: 0.439
Clusters: 3 | Score: 0.374
Clusters: 4 | Score: 0.317
Clusters: 5 | Score: 0.338
Clusters: 6 | Score: 0.350

---Anbefaling---
The best configuration is 2 clusters with a score of 0.439


### Supervised machine learning for classification

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

# Vi laver en kategori: 0 = Billig, 1 = Dyr (baseret på median-prisen)
median_price = cleanDF['price'].median()
cleanDF['price_segment'] = (cleanDF['price'] > median_price).astype(int)

# Definerer variabler (X og y)
X = cleanDF[['milage', 'alder', 'ccm', 'mpg']]
y = cleanDF['price_segment']

# Splitter data i 80% træning og 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Træner en RandomForest model 
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

# Modellen gætter på test-data
y_pred = classifier.predict(X_test)

# Beregner præcision og nøjagtighed
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

# Printer resultaterne
print("--- Classification Resultater ---")
print(f"Accuracy (Rammer rigtigt i alt): {accuracy:.2f}")
print(f"Precision (Pålidelighed ved 'Dyr'): {precision:.2f}")
print(f"Recall (Evne til at finde alle 'Dyre'): {recall:.2f}")
print("\n--- Detaljeret Rapport ---")
print(classification_report(y_test, y_pred, target_names=['Billig', 'Dyr']))

--- Classification Resultater ---
Accuracy (Rammer rigtigt i alt): 0.95
Precision (Pålidelighed ved 'Dyr'): 0.95
Recall (Evne til at finde alle 'Dyre'): 0.94

--- Detaljeret Rapport ---
              precision    recall  f1-score   support

      Billig       0.94      0.95      0.94       273
         Dyr       0.95      0.94      0.95       292

    accuracy                           0.95       565
   macro avg       0.94      0.95      0.95       565
weighted avg       0.95      0.95      0.95       565



### Implementation of the models

In [21]:
import joblib
import pandas as pd
import os

# Opret mappen 'models', hvis den ikke findes i forvejen
if not os.path.exists('models'):
    os.makedirs('models')

# Gem modellerne nede i mappen
joblib.dump(model, 'models/price_model.pkl')
joblib.dump(classifier, 'models/segment_model.pkl')
print("--- Modellerne er gemt i mappen /models ---")

# Hent modellerne frem igen (Husk at skrive stien rigtigt!)
loaded_price_model = joblib.load('models/price_model.pkl')
loaded_segment_model = joblib.load('models/segment_model.pkl')

# Lav en test-bil (Data som modellen aldrig har set før)
test_car = pd.DataFrame([[100000, 5, 1400, 20]], 
                        columns=['milage', 'alder', 'ccm', 'mpg'])

# Brug de hentede modeller
price_prediction = loaded_price_model.predict(test_car)
segment_prediction = loaded_segment_model.predict(test_car)

print(f"\nBilens pris gættes til: {price_prediction[0]:,.0f} kr.")
segment_text = "Dyr (Premium)" if segment_prediction[0] == 1 else "Billig (Budget)"
print(f"Bilen vurderes som: {segment_text}")

--- Modellerne er gemt i mappen /models ---

Bilens pris gættes til: 162,693 kr.
Bilen vurderes som: Billig (Budget)
